##### Импорт библиотек и настройки

In [106]:
# Импорт библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Настройки отображения
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.expand_frame_repr', False)

# Настройки графиков
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 10

# Масштабирование и разделение данных
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

##### Загрузка данных (очищенные, с новым признаком is_new_client)

In [107]:
df = pd.read_csv('../data/processed/result_data.csv')

In [108]:
# Churn - то, что предсказываем. Кодируем ('Yes' - 1, 'No' - 0)
y = df['Churn'].map({'Yes': 1, 'No': 0})

# Признаки
X = df.drop(['Churn', 'customerID', 'Unnamed: 0'], axis=1, errors='ignore')
# Категориальные признаки
cat_cols = X.select_dtypes(exclude='number').columns.to_list()
# Числовые признаки
num_cols = X.select_dtypes(include='number').columns.to_list()

##### ColumnTransformer

In [109]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ],
    remainder='drop'
)

##### Делим на train/test

In [110]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)
print(f"Размер X_train: {X_train.shape}")
print(f"Размер X_test: {X_test.shape}")

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Размер X_train_processed: {X_train_processed.shape}")
print(f"Размер X_test_processed: {X_test_processed.shape}")

Размер X_train: (5634, 20)
Размер X_test: (1409, 20)
Размер X_train_processed: (5634, 46)
Размер X_test_processed: (1409, 46)


##### Сохранение препоцессора и данных

In [111]:
joblib.dump(preprocessor, '../models/preprocessor.joblib')

np.save('../data/processed/X_train_processed.npy', X_train_processed)
np.save('../data/processed/X_test_processed.npy', X_test_processed)
np.save('../data/processed/y_train.npy', y_train.values)
np.save('../data/processed/y_test.npy', y_test.values)